# Sesión 6 · Clasificación
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Problema:** una estación meteorológica alimentada por un panel solar en Anapoima (Cundinamarca) mide, entre muchas variables, el **voltaje del panel (`ws_panel_volt`)**. Ese voltaje indica qué tanta energía está entregando el panel. Vamos a predecir si el voltaje será **ALTO** o **BAJO** a partir de las condiciones del clima.

**Ruta:** datos → variable objetivo → preparación → 3 modelos (con sus hiperparámetros) → evaluación → efecto de los hiperparámetros.

> Ejecute cada celda con `Shift + Enter`. Las celdas de PRÁCTICA son para completar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

pd.set_option('display.max_columns', 30)
print('Librerías cargadas')

## PARTE 1 · Los datos
La estación reporta una lectura cada pocos minutos. El archivo `datos_anapoima.csv` es el histórico completo (jun-2025 a nov-2026) y está publicado en GitHub, así que lo leemos directamente desde la URL.

In [ ]:
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_anapoima.csv'
df = pd.read_csv(url, parse_dates=['fecha_hora'])
print(f'{df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

## PARTE 2 · ¿Qué mide el voltaje del panel?
`ws_panel_volt` es el voltaje en los bornes del panel solar de la estación. De noche el panel no produce y el voltaje cae a ~0 V; de día sube hasta ~13-14 V cuando carga la batería. Miremos su comportamiento a lo largo del día.

In [ ]:
por_hora = df.groupby(df['fecha_hora'].dt.hour)['ws_panel_volt'].mean()
por_hora.plot(kind='bar', figsize=(9, 3.5), color='tab:orange', edgecolor='black')
plt.title('Voltaje medio del panel por hora del día')
plt.xlabel('Hora'); plt.ylabel('Voltaje (V)'); plt.show()

### PRÁCTICA 1


1. ¿Cuántas columnas tiene el dataset y cuáles son?
2. ¿Cuál es el rango de fechas?
3. ¿Cuántos valores faltantes hay en `ws_panel_volt`?

*Pista:* `df.columns`, `df['fecha_hora'].min()`, `df['ws_panel_volt'].isna().sum()`.

In [ ]:
print(df.columns.tolist())
print(df['fecha_hora'].min(), '->', df['fecha_hora'].max())
print('Faltantes en ws_panel_volt:', df['ws_panel_volt'].isna().sum())

## PARTE 3 · Limpieza mínima
Dos decisiones antes de modelar:
1. **Quitar lecturas repetidas:** el sistema a veces repite la misma marca de tiempo; nos quedamos con una por `fecha_hora`.
2. **Quedarnos solo con el día** (`irradiancia_wm2 > 50`). De noche el voltaje es ~0 y adivinarlo sería trivial; lo interesante es predecir cuánto voltaje entrega el panel *cuando hay sol*.

In [ ]:
print('Antes:', len(df))
df = df.drop_duplicates(subset='fecha_hora')
df = df[df['irradiancia_wm2'] > 50]
print('Después:', len(df))
df['ws_panel_volt'].describe().round(2)

## PARTE 4 · Convertir el voltaje en una clase
Clasificar es predecir una **categoría**, no un número. Convertimos el voltaje en dos clases usando la **mediana** como umbral (así quedan ~50 % y ~50 %):
- **BAJO (0):** voltaje ≤ mediana
- **ALTO (1):** voltaje > mediana

*Regresión* sería predecir "13.1 V"; *clasificación* es predecir "ALTO".

In [ ]:
umbral = df['ws_panel_volt'].median()
df['volt_clase'] = (df['ws_panel_volt'] > umbral).astype(int)
df['volt_etiqueta'] = df['volt_clase'].map({0: 'BAJO', 1: 'ALTO'})
print(f'Umbral (mediana): {umbral:.2f} V')
print(df['volt_etiqueta'].value_counts())

df['ws_panel_volt'].plot(kind='hist', bins=50, figsize=(8, 3.5), edgecolor='black')
plt.axvline(umbral, color='red', linestyle='--', label=f'umbral = {umbral:.2f} V')
plt.title('Distribución del voltaje del panel (de día)'); plt.xlabel('Voltaje (V)'); plt.legend(); plt.show()

### PRÁCTICA 2


Pruebe otro umbral: use el **percentil 75** en lugar de la mediana.
1. Calcule el percentil 75 del voltaje.
2. Cree una clase binaria con ese umbral.
3. ¿Qué porcentaje queda en cada clase? ¿Está balanceado? ¿Por qué importa?

*Pista:* `df['ws_panel_volt'].quantile(0.75)`, `value_counts(normalize=True)`.

In [ ]:
p75 = df['ws_panel_volt'].quantile(0.75)
clase_p75 = (df['ws_panel_volt'] > p75).astype(int)
print(f'Percentil 75: {p75:.2f} V')
print((clase_p75.value_counts(normalize=True) * 100).round(1))
print('Con p75 quedan ~75 % / 25 %: clases desbalanceadas, el modelo tiende a favorecer la clase mayoritaria.')

## PARTE 5 · ¿Con qué variables lo predecimos?
Usaremos 5 variables del clima como entrada (**features**, `X`) y la clase de voltaje como respuesta (**target**, `y`). Antes de modelar, revisamos cuáles se relacionan con el voltaje.

In [ ]:
features = ['temperatura_c', 'irradiancia_wm2', 'velocidad_viento_kmh', 'humedad_relativa', 'presion_hpa']
df_modelo = df[features + ['ws_panel_volt', 'volt_clase']].dropna()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(df_modelo[features + ['ws_panel_volt']].corr().round(2), annot=True, cmap='coolwarm', ax=ax[0])
ax[0].set_title('Correlación')
ax[1].scatter(df_modelo['irradiancia_wm2'], df_modelo['ws_panel_volt'], s=3, alpha=0.3)
ax[1].set_xlabel('Irradiancia (W/m2)'); ax[1].set_ylabel('Voltaje (V)'); ax[1].set_title('Irradiancia vs voltaje')
plt.tight_layout(); plt.show()

### PRÁCTICA 3


1. ¿Qué variable tiene la correlación más alta (en valor absoluto) con `ws_panel_volt`?
2. ¿Tiene sentido físico? Escriba su explicación en un comentario.

*Pista:* `df_modelo.corr()['ws_panel_volt'].abs().sort_values()`.

In [ ]:
corr = df_modelo.drop(columns='volt_clase').corr()['ws_panel_volt'].drop('ws_panel_volt')
print(corr.abs().sort_values(ascending=False).round(2))
# Más irradiancia -> más voltaje del panel; la humedad y el viento influyen de forma indirecta (nubosidad, enfriamiento).

## PARTE 6 · Dividir y escalar
- **Train/test (80/20):** entrenamos con una parte y evaluamos con datos que el modelo nunca vio. `stratify=y` conserva la proporción de clases en ambos grupos.
- **Escalado (`StandardScaler`):** deja cada variable con media 0 y desviación 1. Importa para regresión logística y SVM (usan distancias/pesos); el Random Forest no lo necesita pero no le afecta.
- El escalador se ajusta (**`fit`**) **solo con train**, para no filtrar información del test.

In [ ]:
X = df_modelo[features]
y = df_modelo['volt_clase']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print(f'Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas')
print('Proporción ALTO en train:', round(y_train.mean(), 3), '| en test:', round(y_test.mean(), 3))

In [ ]:
def evaluar(modelo, nombre, Xtr=None, Xte=None):
    Xtr = X_train_s if Xtr is None else Xtr
    Xte = X_test_s if Xte is None else Xte
    pred = modelo.predict(Xte)
    return {'Modelo': nombre,
            'Acc train': round(accuracy_score(y_train, modelo.predict(Xtr)), 3),
            'Acc test': round(accuracy_score(y_test, pred), 3),
            'Precision': round(precision_score(y_test, pred), 3),
            'Recall': round(recall_score(y_test, pred), 3),
            'F1': round(f1_score(y_test, pred), 3)}

## PARTE 7 · Modelo 1: Regresión Logística
Calcula una combinación lineal de las variables y la convierte en una probabilidad (curva sigmoide). Si la probabilidad de ALTO > 0.5, predice ALTO. Su frontera de decisión es una **línea/plano**: simple, rápida e interpretable.

### Hiperparámetros principales
| Hiperparámetro | Qué controla | Si lo aumentas / cambias |
|---|---|---|
| `C` (def. 1.0) | **Inverso** de la regularización: qué tan libre es el modelo de ajustar los datos | `C` grande → regularización débil, pesos más grandes, riesgo de sobreajuste. `C` pequeño → modelo más "frenado", puede quedar corto (subajuste) |
| `penalty` (`'l2'`, `'l1'`) | Tipo de castigo a los pesos | `l2` los encoge; `l1` lleva algunos a exactamente 0 (selecciona variables). `l1` requiere `solver='liblinear'` o `'saga'` |
| `solver` | Algoritmo de optimización | Cambia velocidad y qué penalizaciones admite, no la idea del modelo |
| `max_iter` (def. 100) | Máximo de iteraciones del optimizador | Muy bajo → advertencia de "no convergió" y modelo a medias |
| `class_weight` (`None`/`'balanced'`) | Peso de cada clase | `'balanced'` ayuda cuando una clase es minoritaria (mejora recall de esa clase) |

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42).fit(X_train_s, y_train)
print(evaluar(lr, 'Reg. Logística'))
print('\nPesos (importancia con signo) por variable:')
print(pd.Series(lr.coef_[0], index=features).round(2))

### PRÁCTICA 4


Entrene la regresión logística con `C` = 0.001, 0.01, 1 y 100. Para cada uno imprima la precisión en train y en test.
¿Cambia mucho el resultado? ¿Por qué cree que ocurre (recuerde cuántas variables y cuántos datos hay)?

*Pista:* un `for C in [...]` y `evaluar(modelo, nombre)`.

In [ ]:
filas = []
for C in [0.001, 0.01, 1, 100]:
    m = LogisticRegression(C=C, max_iter=1000, random_state=42).fit(X_train_s, y_train)
    filas.append(evaluar(m, f'LR C={C}'))
display(pd.DataFrame(filas))
# Con solo 5 variables y >20.000 datos, el modelo apenas puede sobreajustar: C cambia poco.
# Solo con C muy pequeño (0.001) se nota un ligero subajuste.

## PARTE 8 · Modelo 2: Random Forest
Entrena muchos **árboles de decisión** (cada uno ve una muestra distinta de filas y de variables) y decide por **votación**. Captura relaciones no lineales sin necesitar escalado.

### Hiperparámetros principales
| Hiperparámetro | Qué controla | Si lo aumentas |
|---|---|---|
| `n_estimators` (def. 100) | Número de árboles | Más árboles → resultado más estable y mejor, con rendimientos decrecientes; cuesta más tiempo |
| `max_depth` (def. `None`) | Profundidad máxima de cada árbol | **Es el que más pesa.** Poco profundo → subajuste (reglas muy simples). Sin límite → árboles memorizan el train (sobreajuste) |
| `min_samples_leaf` (def. 1) | Mínimo de datos por hoja | Más alto → hojas más "promedio", modelo más suave y menos sobreajustado |
| `min_samples_split` (def. 2) | Mínimo de datos para dividir un nodo | Más alto → árboles más pequeños |
| `max_features` (def. `'sqrt'`) | Cuántas variables mira cada división | Menos → árboles más distintos entre sí (más diversidad); más → árboles más parecidos |
| `class_weight` | Peso de cada clase | `'balanced'` si hay clases desbalanceadas |
| `random_state` | Semilla | No mejora el modelo: hace el resultado **reproducible** |

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=1, random_state=42, n_jobs=-1).fit(X_train_s, y_train)
print(evaluar(rf, 'Random Forest'))
print('\nImportancia de variables:')
print(pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False).round(3))

### PRÁCTICA 5


Entrene Random Forest variando `max_depth` en 2, 5, 10 y `None`. Para cada uno muestre precisión en **train** y **test**.
1. ¿Con qué profundidad el modelo "memoriza" (train mucho mayor que test)?
2. ¿Con cuál queda corto (subajuste)?

*Pista:* la diferencia `Acc train - Acc test` es la señal de sobreajuste.

In [ ]:
filas = []
for d in [2, 5, 10, None]:
    m = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42, n_jobs=-1).fit(X_train_s, y_train)
    filas.append(evaluar(m, f'RF max_depth={d}'))
res = pd.DataFrame(filas)
res['Brecha'] = (res['Acc train'] - res['Acc test']).round(3)
display(res)
# max_depth=2: train y test bajos y parecidos (subajuste). Sin límite: train ~1.0 y test menor (sobreajuste).

## PARTE 9 · Modelo 3: SVM (Máquina de Vectores de Soporte)
Busca la **frontera que separa las clases con el mayor margen** posible. Con un *kernel* puede trazar fronteras curvas. Es sensible a la escala (por eso escalamos) y se vuelve lento con muchos datos.

### Hiperparámetros principales
| Hiperparámetro | Qué controla | Si lo aumentas / cambias |
|---|---|---|
| `kernel` (`'linear'`, `'rbf'`, `'poly'`) | Forma de la frontera | `linear` → recta/plano; `rbf` → curva flexible (el más usado); `poly` → polinomial |
| `C` (def. 1.0) | Cuánto se castigan los errores de entrenamiento | `C` grande → margen estrecho, intenta no fallar ningún punto (riesgo de sobreajuste). `C` pequeño → margen amplio, tolera errores (más suave) |
| `gamma` (def. `'scale'`, solo rbf/poly) | Alcance de influencia de cada punto | `gamma` grande → influencia local, frontera muy "enroscada" (sobreajuste). `gamma` pequeño → frontera casi recta (subajuste) |
| `class_weight` | Peso de cada clase | `'balanced'` para clases desbalanceadas |
| `probability` | Si calcula probabilidades | `True` las da pero entrena más lento |

In [ ]:
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_train_s, y_train)
print(evaluar(svm, 'SVM rbf'))

### PRÁCTICA 6


Entrene SVM (kernel `rbf`) con `gamma` = 0.01, 0.1, 1 y 10 (manteniendo `C=1`). ¿Qué pasa con la precisión de train y de test cuando `gamma` crece?

*Nota:* el SVM tarda unos segundos por modelo.

In [ ]:
filas = []
for g in [0.01, 0.1, 1, 10]:
    m = SVC(kernel='rbf', C=1.0, gamma=g).fit(X_train_s, y_train)
    filas.append(evaluar(m, f'SVM gamma={g}'))
res = pd.DataFrame(filas)
res['Brecha'] = (res['Acc train'] - res['Acc test']).round(3)
display(res)
# gamma pequeño: frontera casi lineal. gamma grande: train sube y test baja (sobreajuste).

## PARTE 10 · Comparar los modelos
Cuatro métricas, todas calculadas sobre el conjunto de **test**:
- **Accuracy:** de todas las predicciones, ¿qué fracción acertó?
- **Precision:** de lo que predije ALTO, ¿cuánto era realmente ALTO?
- **Recall:** de lo que era realmente ALTO, ¿cuánto detecté?
- **F1:** promedio armónico de precision y recall.

In [ ]:
tabla = pd.DataFrame([evaluar(lr, 'Reg. Logística'), evaluar(rf, 'Random Forest'), evaluar(svm, 'SVM rbf')]).set_index('Modelo')
display(tabla)
tabla[['Acc test', 'Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(9, 4), ylim=(0.5, 1.0), rot=0)
plt.title('Comparación de modelos (test)'); plt.legend(loc='lower right'); plt.show()

### PRÁCTICA 7


1. ¿Qué modelo tiene mejor `Acc test`? ¿Y mejor `F1`?
2. ¿Alguno muestra una brecha grande entre `Acc train` y `Acc test`? ¿Qué significa?

In [ ]:
print('Mejor Acc test:', tabla['Acc test'].idxmax())
print('Mejor F1:', tabla['F1'].idxmax())
print((tabla['Acc train'] - tabla['Acc test']).round(3))
# Un Random Forest sin límite de profundidad suele tener train ~1.0: memoriza; la brecha con test indica sobreajuste.

## PARTE 11 · Matriz de confusión
Muestra **dónde** se equivoca cada modelo:

|  | Predicho BAJO | Predicho ALTO |
|---|---|---|
| **Real BAJO** | VN (acierto) | FP (falsa alarma) |
| **Real ALTO** | FN (se escapó) | VP (acierto) |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, m, cmap) in zip(axes, [('Reg. Logística', lr, 'Blues'), ('Random Forest', rf, 'Greens'), ('SVM rbf', svm, 'Oranges')]):
    cm = confusion_matrix(y_test, m.predict(X_test_s))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax, xticklabels=['BAJO', 'ALTO'], yticklabels=['BAJO', 'ALTO'])
    ax.set_title(nombre); ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()

### PRÁCTICA 8


Con la matriz del Random Forest (`cm_rf`), obtenga VN, FP, FN y VP y calcule a mano el **recall** (VP / (VP + FN)). Compare con el valor de la tabla.

*Pista:* `VN, FP, FN, VP = cm_rf.ravel()`.

In [ ]:
cm_rf = confusion_matrix(y_test, rf.predict(X_test_s))
VN, FP, FN, VP = cm_rf.ravel()
print(f'VN={VN}  FP={FP}  FN={FN}  VP={VP}')
print('Recall a mano:', round(VP / (VP + FN), 3), '| en la tabla:', tabla.loc['Random Forest', 'Recall'])

## PARTE 12 · Ver el efecto de un hiperparámetro
Dibujamos la precisión de **train** y **test** del Random Forest al aumentar `max_depth`:
- Zona izquierda (árboles muy simples): ambas bajas → **subajuste**.
- Zona derecha (árboles muy profundos): train sube a ~1 y test deja de mejorar o baja → **sobreajuste**.
- El mejor punto está donde el test es máximo.

Esa es la idea central de **ajustar hiperparámetros**: buscar el equilibrio entre un modelo muy simple y uno que memoriza.

In [ ]:
profundidades = [1, 2, 3, 5, 8, 12, 16, 20]
acc_tr, acc_te = [], []
for d in profundidades:
    m = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42, n_jobs=-1).fit(X_train_s, y_train)
    acc_tr.append(m.score(X_train_s, y_train)); acc_te.append(m.score(X_test_s, y_test))

plt.figure(figsize=(8, 4))
plt.plot(profundidades, acc_tr, 'o-', label='Train'); plt.plot(profundidades, acc_te, 's-', label='Test')
plt.xlabel('max_depth'); plt.ylabel('Accuracy'); plt.title('Random Forest: efecto de max_depth')
plt.grid(alpha=0.3); plt.legend(); plt.show()
print('Mejor max_depth en test:', profundidades[int(np.argmax(acc_te))])

## Resumen
- Clasificación = predecir una **categoría**; aquí, voltaje del panel **ALTO/BAJO** a partir del clima.
- Convertimos una variable continua en clase con un umbral (mediana) y revisamos el balance.
- Tres modelos y sus hiperparámetros clave: **LR** (`C`), **Random Forest** (`max_depth`, `n_estimators`, `min_samples_leaf`), **SVM** (`C`, `gamma`, `kernel`).
- Un hiperparámetro mal elegido produce **subajuste** o **sobreajuste**; se detecta comparando train vs test.
- Se evalúa con accuracy, precision, recall, F1 y la matriz de confusión.

**Ejercicio final:** cambie el umbral de la PARTE 4 (por ejemplo a 13 V) o agregue la hora del día como variable, y repita la comparación. ¿Qué modelo se ve más afectado?